In [1]:
import pandas as pd
import re

def read_news_to_dataframe(file_path):
    """
    读取新闻文本文件并将每篇新闻报道分隔开来，返回一个 DataFrame。
    
    :param file_path: 新闻文本文件路径
    :return: 包含每篇新闻报道的 DataFrame
    """
    # 读取文件内容
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()

    # 使用正则表达式按照一个或多个空行分隔新闻报道
    news_list = re.split(r'\n\s*\n', content.strip())  # \n\s*\n 匹配一个或多个空行

    # 创建 DataFrame
    df = pd.DataFrame(news_list, columns=["text"])

    return df

# 假设文件路径是 'news.txt'
file_path = '/hongyi/stream/PMI/dataset/News-Commentary.zh'

# 调用函数
df = read_news_to_dataframe(file_path)
df['labels'] = range(len(df))
# 打印 DataFrame
print(df.head())  # 打印前几行新闻报道


                                                text  labels
0  1929年还是1989年? \n巴黎-随着经济危机不断加深和蔓延，整个世界一直在寻找历史上的...       0
1  百年愚顽 \n柏林 — — 2008年爆发的全球金融和经济危机是自大萧条以来最严峻的一次经济...       1
2  2008年败在何处？ \n伯克利—要解决问题，光知道做什么是不够的。 你实际上必须实施解决办...       2
3  欧洲的重振战略 \n斯托克霍姆/马德里—去年11月，教皇方济各在其欧洲议会演讲中将欧盟比作祖...       3
4  标志着时代终结的一年？ \n马德里 — — 随着2016年的结束，2017年的前景被不确定性...       4


In [5]:
df['text'][0]

'1929年还是1989年? \n巴黎-随着经济危机不断加深和蔓延，整个世界一直在寻找历史上的类似事件希望有助于我们了解目前正在发生的情况。 一开始，很多人把这次危机比作1982年或1973年所发生的情况，这样得类比是令人宽心的，因为这两段时期意味着典型的周期性衰退。 \n如今人们的心情却是沉重多了，许多人开始把这次危机与1929年和1931年相比，即使一些国家政府的表现仍然似乎把视目前的情况为是典型的而看见的衰退。 目前的趋势是，要么是过度的克制（欧洲 ） ， 要么是努力的扩展（美国 ） 。 欧洲在避免债务和捍卫欧元的名义下正变得谨慎，而美国已经在许多方面行动起来，以利用这一理想的时机来实行急需的结构性改革。 \n然而，作为地域战略学家，无论是从政治意义还是从经济意义上，让我自然想到的年份是1989年。 当然，雷曼兄弟公司的倒闭和柏林墙的倒塌没有任何关系。 事实上，从表面上看，两者似乎是完全是相反的：一个是象征着压抑和人为分裂的柏林墙的倒塌，而另一个是看似坚不可摧的并令人安心的金融资本主义机构的倒塌。 \n然而，和1989年一样，2008-2009年很可能也能被视为一个划时代的改变，其带来的发人深省的后果将在几十年后仍能让我们感受得到。 东西方意识形态鸿沟的结束，以及对市场绝对信心的后果，都是历史的转折点。 而2009年所发生的事情可能会威胁1989年革命所带来的积极成果，包括欧洲的和平统一和民主制度战胜了民族主义倾向，如果不是恐外倾向的话。 \n1989年，自由民主战胜了由苏联集团具体化并推崇的社会主义意识形态。 对于里根总统的许多的支持者来说，就是他精心策划的军备竞赛的升级，把苏联经济推向了崩溃的边缘，从而充分显示了自由社会和自由市场的优越性。 \n当然，现在的情况和1989年的情况明显不同了。 首先，也许是最重要的，1989年的革命和随后的苏联解体结束了全球的两极化。 与此相反，2009年很可能会为一种新的两极化形式铺平道路，只是中国取代了苏联。 \n其二，民主制度和市场资本主义，或许要比预期的要脆弱些，看来确实是当时的赢家。 而在2009年，随着全球危机的蔓延，却很难区分赢家和输家；每个人似乎都是输家，即使有些国家比其它国家受到的影响更大。 \n而历史是不公平的。 尽管美国要为当今的全球危机负更大的责任，但美国可能会比大多数国家以更良好的势态走出困境。 美

In [2]:
from collections import defaultdict

def build_index_mapping(documents):
    """
    构建词索引映射，用于快速查询每个词在文档中的位置。
    
    Args:
        documents (list of list): 数据集，每个文档是一个分词后的词列表。
    
    Returns:
        list of dict: 每个文档中词到位置列表的映射。
    """
    index_mappings = []
    for doc in documents:
        index_map = defaultdict(list)
        for i, word in enumerate(doc):
            index_map[word].append(i)
        index_mappings.append(index_map)
    return index_mappings


def count_nonoverlapping_intervals(index_mappings, word_pairs):
    """
    计算给定词对在每个文档中的非重叠出现次数。
    
    Args:
        index_mappings (list of dict): 每个文档的词索引映射。
        word_pairs (list of tuple): 要查询的词对列表，每个元素是 (word1, word2)。
    
    Returns:
        dict: 每个词对的非重叠出现次数列表。
    """
    results = {pair: [] for pair in word_pairs}
    
    for index_map in index_mappings:
        for word1, word2 in word_pairs:
            positions1 = sorted(index_map[word1])
            positions2 = sorted(index_map[word2])
            
            if not positions1 or not positions2:
                results[(word1, word2)].append(0)
                continue
            
            # 确定哪个词先出现
            if positions1[0] <= positions2[0]:
                positions_a, positions_b = positions1, positions2
            else:
                positions_a, positions_b = positions2, positions1
            
            # 计算非重叠区间
            count = 0
            last_start, last_end = -1, -1  # 上一个区间的起始和结束位置
            i, j = 0, 0  # i 是 A 的位置索引，j 是 B 的位置索引
            
            while i < len(positions_a) and j < len(positions_b):
                # 确定当前 A 和 B 的位置
                a, b = positions_a[i], positions_b[j]
                
                # A 和 B 的顺序决定新的区间
                if a < b:
                    if a > last_end and b > last_end:  # 不在上一个区间中
                        count += 1
                        last_start, last_end = a, b
                        i += 1
                        j += 1
                    else:
                        i += 1  # 跳过当前 A
                else:  # 如果 B 在 A 前面，调整顺序
                    if b > last_end and a > last_end:
                        count += 1
                        last_start, last_end = b, a
                        i += 1
                        j += 1
                    else:
                        j += 1  # 跳过当前 B
            
            results[(word1, word2)].append(count)
    
    return results


# # 示例数据集
# documents = [
#     ["我", "在", "处理", "大", "自然", "中", "使用", "自然", "语言", "处理", "方法", "来", "处理", "问题"],
#     ["我", "在", "处理", "大", "自然", "中", "使用", "自然", "语言", "处理", "方法"],
#     ["自然", "语言", "中", "使用", "处理", "方法", "自然", "处理", "语言"]
# ]

# # 构建词索引映射
# index_mappings = build_index_mapping(documents)

# # 查询的词对
# word_pairs = [("自然", "处理"),("处理", "自然")]

# # 计算非重叠出现次数
# results = count_nonoverlapping_intervals(index_mappings, word_pairs)

# # 打印结果
# for pair, counts in results.items():
#     print(f"词对 {pair}: {counts}")


In [3]:
from collections import defaultdict

def count_nonoverlapping_intervals_with_threshold(index_mappings, word_pairs, threshold):
    """
    计算给定词对在每个文档中的非重叠出现次数，考虑间隔单词数阈值。

    Args:
        index_mappings (list of dict): 每个文档的词索引映射。
        word_pairs (list of tuple): 要查询的词对列表，每个元素是 (word1, word2)。
        threshold (int): A 和 B 之间允许的最大单词间隔。

    Returns:
        dict: 每个词对的非重叠出现次数列表。
    """
    results = {pair: [] for pair in word_pairs}

    for index_map in index_mappings:
        for word1, word2 in word_pairs:
            positions1 = sorted(index_map[word1])
            positions2 = sorted(index_map[word2])

            if not positions1 or not positions2:
                results[(word1, word2)].append(0)
                continue

            # 确定哪个词先出现
            if positions1[0] <= positions2[0]:
                positions_a, positions_b = positions1, positions2
            else:
                positions_a, positions_b = positions2, positions1

            # 计算非重叠区间
            count = 0
            last_start, last_end = -1, -1  # 上一个区间的起始和结束位置
            i, j = 0, 0  # i 是 A 的位置索引，j 是 B 的位置索引

            while i < len(positions_a) and j < len(positions_b):
                # 确定当前 A 和 B 的位置
                a, b = positions_a[i], positions_b[j]

                # 检查间隔是否满足阈值
                if abs(a - b) - 1 > threshold:
                    if a < b:
                        i += 1  # 跳过当前 A
                    else:
                        j += 1  # 跳过当前 B
                    continue

                # 确定 A 和 B 的顺序是否构成有效区间
                if a > last_end and b > last_end:  # 不在上一个区间中
                    count += 1
                    last_start, last_end = a, b
                    i += 1
                    j += 1
                else:
                    # 跳过被覆盖的 A 或 B
                    if a <= last_end:
                        i += 1
                    if b <= last_end:
                        j += 1

            results[(word1, word2)].append(count)

    return results


# # 示例数据集
# documents = [
#     ["我", "在", "大", "处理","我", "在", "大", "自然", "中", "使用", "自然", "语言", "处理", "方法", "来", "处理", "问题"],
#     ["我", "在", "大", "处理", "自然", "中", "使用", "自然", "语言", "处理", "方法"],
#     ["自然", "语言","我", "在", "大", "处理", "方法", "自然", "处理", "语言", "自然"]
# ]

# # 查询的词对
# word_pairs = [("自然", "处理"), ("处理", "自然")]

# # 设置单词间隔阈值
# threshold = 2

# # 计算非重叠出现次数
# results = count_nonoverlapping_intervals_with_threshold(index_mappings, word_pairs, threshold)

# # 打印结果
# for pair, counts in results.items():
#     print(f"词对 {pair}: {counts}")


In [1]:
import pandas as pd
import json

# 从文件中读取JSON数据
with open('/hongyi/stream/PMI/dataset/MultiUN_zh.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# 创建DataFrame，列名为'text'
df = pd.DataFrame(data, columns=['text'])
df['labels'] = range(len(df))
# 显示DataFrame
print(df.head())

                                                text  labels
0                                   2005年5月2日至27日，纽约       0
1  马来西亚代表团在不扩散核武器条约缔约国2005年审议大会全体会议上以不扩散核武器条约不结盟缔...       1
2           1. 不扩散条约不结盟缔约国欢迎通过了不扩散条约缔约国2005年审议大会的议程。       2
3  议程建立了一个框架，以便按照本条约第八条第3款、过去历次审议大会（特别是1995年审议大会和...       3
4  2. 不扩散条约不结盟缔约国重申它们关于将本着诚意履行其根据本条约所承担义务的承诺，并重申在...       4


In [4]:

#本段落用时9min
from stream_topic.utils.dataset import TMDataset

# 创建 TMDataset 实例
dataset = TMDataset()#language="zh-cn", stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt'

# 假设你有一个名为 df 的 DataFrame，包含你的数据
# df = pd.DataFrame(...)

# 数据集名称
dataset_name = "news_zh_char"

# 指定保存数据集的目录
save_dir = "/hongyi/stream/PMI/dataset"
# df = df.iloc[0:100]
# 调用 create_load_save_dataset 方法来存储数据集
dataset.create_load_save_dataset(
    data=df,
    dataset_name=dataset_name,
    save_dir=save_dir,
    doc_column="text",  # 假设 DataFrame 中包含文本的列名为 "text_column"
    label_column="labels",  # 假设 DataFrame 中包含标签的列名为 "label_column"
    language = "chinese",
    stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt',
    min_word_length = 1
)#, min_word_freq=1

# 打印保存的数据集信息
print(dataset.dataframe.head())


/hongyi/anaconda3/envs/pku/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
Preprocessing documents: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7888/7888 [03:02<00:00, 43.33it/s]
2025-02-10 10:45:47.655 | INFO     | stream_topic.utils.dataset:create_load_save_dataset:239 - Dataset saved to /hongyi/stream/PMI/dataset/news_zh_char.parquet
2025-02-10 10:45:47.899 | INFO     | stream_topic.utils.dataset:create_load_save_dataset:254 - Dataset info saved to /hongyi/stream/PMI/dataset/news_zh_char_info.pkl


  language                                stopwords_path  min_word_length  \
0  chinese  /hongyi/stream/stopwords/baidu_stopwords.txt                1   
1  chinese  /hongyi/stream/stopwords/baidu_stopwords.txt                1   
2  chinese  /hongyi/stream/stopwords/baidu_stopwords.txt                1   
3  chinese  /hongyi/stream/stopwords/baidu_stopwords.txt                1   
4  chinese  /hongyi/stream/stopwords/baidu_stopwords.txt                1   

                                                text  labels  
0  年 还 年 济 危 机 不 深 蔓 延 世 界 直 历 史 上 类 似 事 希 解 目 前 ...       0  
1  年 柏 林 年 爆 发 全 球 金 融 济 危 机 大 萧 条 最 严 次 济 压 力 二 ...       1  
2  年 败 处 克 利 解 决 问 题 知 道 做 什 不 够 实 际 上 必 须 实 施 解 ...       2  
3  欧 洲 重 振 战 略 斯 克 马 德 里 去 年 月 教 皇 方 济 欧 洲 议 会 演 ...       3  
4  标 志 时 代 终 结 年 马 德 里 年 结 束 年 前 景 不 确 定 性 笼 罩 中 ...       4  


In [5]:
contains_english = dataset.dataframe['text'].str.contains(r'[A-Za-z]', na=False)

# 输出包含英文字母的行
english_rows = dataset.dataframe[contains_english]
print(english_rows)

Empty DataFrame
Columns: [language, stopwords_path, min_word_length, text, labels]
Index: []


In [1]:
from stream_topic.models import KmeansTM,BERTopicTM,CBC,DCTE,NMFTM,SOMTM,CEDC,ETM,LDA,ProdLDA,SOMTM,NSTM,WordCluTM,CTM,TNTM,NeuralLDA,CTMNeg
from stream_topic.utils import TMDataset
#本段落用时9min
dataset = TMDataset(language="chinese", stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt')# 
dataset.fetch_dataset(name = "news_zh_jieba", dataset_path = "/hongyi/stream/PMI/dataset", source = 'local')
dataset.preprocess(model_type="KmeansTM", min_word_length = 1)
#本段落用时4h
model = KmeansTM(embedding_model_name="/hongyi/stream/sentence-transformers/Conan-embedding-v1/",stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt')#
model.fit(dataset,n_topics=10)
# model = NMFTM(stopwords_path = '/hongyi/stream/stopwords/scu_stopwords.txt')# 
# model.fit(dataset)#

topics = model.get_topics()
print(topics)

/hongyi/anaconda3/envs/mystream/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-03-07 14:42:05.058 | INFO     | stream_topic.utils.dataset:fetch_dataset:121 - Fetching dataset: news_zh_jieba
2025-03-07 14:42:05.059 | INFO     | stream_topic.utils.dataset:fetch_dataset:131 - Fetching dataset from local path
Preprocessing documents:   0%|                                                                                                                                 | 0/7888 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.527 seconds.
Prefix dict has been built successfully.
Preprocessing documents:  21%|████████████████████████▌                                                  

KeyboardInterrupt: 

In [ ]:
[['普京', '乌克兰', '移民', '奥巴马', '波兰', '北约', '难民', '民粹主义', '苏联', '法国']
 , ['互联网', '税收', '新闻', '创新', '网络', '信息', '数据', '服务', '平等', '工人']
 , ['气候', '排放', '燃料', '能源', '二氧化碳', '化石', '变暖', '石油', '地球', '再生']
 , ['朝鲜', '韩国', '墨西哥', '习近平', '亚洲', '缅甸', '关税', '金正', '外交', '委内瑞拉']
 , ['巴勒斯坦', '以色列', '伊斯兰', '阿拉伯', '埃及', '叙利亚', '伊拉克', '土耳其', '沙特', '伊朗']
 , ['美联储', '普京', '奥巴马', '资产', '人民币', '汇率', '新兴', '贷款', '债券', '储备']
 , ['疾病', '女性', '治疗', '儿童', '妇女', '疫苗', '基因', '药物', '卫生', '患者']
 , ['希腊', '欧元区', '欧元', '意大利', '央行', '联盟', '西班牙', '债券', '紧缩', '成员国']]

In [1]:
from stream_topic.models import KmeansTM,BERTopicTM,CBC,DCTE,NMFTM,SOMTM,CEDC,ETM,LDA,ProdLDA,SOMTM,NSTM,WordCluTM,CTM,TNTM,NeuralLDA,CTMNeg
from stream_topic.utils import TMDataset
#本段落用时9min
dataset = TMDataset(language="chinese", stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt')# 
dataset.fetch_dataset(name = "cnews_test", dataset_path = "/hongyi/stream/dataset/processed_dataset", source = 'local')
dataset.preprocess(model_type="KmeansTM", min_word_length = 1)
#本段落用时4h
model = KmeansTM(embedding_model_name="/hongyi/stream/sentence-transformers/Conan-embedding-v1/",stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt')#
model.fit(dataset,n_topics=10)
# model = NMFTM(stopwords_path = '/hongyi/stream/stopwords/scu_stopwords.txt')# 
# model.fit(dataset)#

topics = model.get_topics()
print(topics)

/hongyi/anaconda3/envs/pku/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-02-06 15:32:25.766 | INFO     | stream_topic.utils.dataset:fetch_dataset:120 - Fetching dataset: cnews_test
2025-02-06 15:32:25.768 | INFO     | stream_topic.utils.dataset:fetch_dataset:130 - Fetching dataset from local path
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.539 seconds.
Prefix dict has been built successfully.
Preprocessing documents: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [

[['搭配', '导语', '性感', '时尚', '黑色', '组图', '外套', '白色', '款式', '单品'], ['导演', '电影', '影片', '观众', '票房', '演员', '娱乐', '微博', '上映', '角色'], ['房地产', '房价', '土地', '楼市', '开发商', '地产', '住房', '楼盘', '平方米', '成交'], ['股票', '型基金', '债券', '指数', '收益', '分红', '经理', '投资者', '仓位', '净值'], ['比赛', '球队', '篮板', '球员', '季后赛', '赛季', '热火', '火箭', '进攻', '防守'], ['留学', '移民', '学生', '大学', '学校', '申请', '签证', '留学生', '考试', '教育'], ['像素', '机身', '英寸', '采用', '佳能', '功能', '光学', '相机', '索尼', '高清'], ['玩家', '游戏', '手机', '网游', '奖励', '装备', '客服', '活动', '封神', '手机游戏'], ['家具', '家居', '地板', '消费者', '品牌', '装修', '家装', '橱柜', '卖场', '建材'], ['台湾', '马英九', '主席', '合作', '陈水扁', '胡锦涛', '两岸', '中方', '日电', '会议']]


In [3]:
from stream_topic.metrics import ISIM, INT, ISH,Expressivity, NPMI,PMI,cPMI, Embedding_Coherence, Embedding_Topic_Diversity
from sentence_transformers import SentenceTransformer
from stream_topic.metrics.metrics_config import MetricsConfig
MetricsConfig.set_PARAPHRASE_embedder("/hongyi/stream/sentence-transformers/Conan-embedding-v1/")#paraphrase-multilingual-mpnet-base-v2
MetricsConfig.set_SENTENCE_embedder("/hongyi/stream/sentence-transformers/Conan-embedding-v1/")

In [4]:
import pandas as pd
def load_stopwords(stopwords_path):
        # load Chinese stopwords list
        return pd.read_csv(stopwords_path, names=['w'], sep='\t', encoding='UTF-8')
stopword = load_stopwords('/hongyi/stream/stopwords/baidu_stopwords.txt')

In [23]:
from stream_topic.models import KmeansTM,BERTopicTM,CBC,DCTE,NMFTM,SOMTM,CEDC,ETM,LDA,ProdLDA,SOMTM,NSTM,WordCluTM,CTM,TNTM,NeuralLDA,CTMNeg
from stream_topic.utils import TMDataset
#本段落用时9min
dataset = TMDataset(language="chinese", stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt')# 
dataset.fetch_dataset(name = "news_zh_hanlp", dataset_path = "/hongyi/stream/PMI/dataset", source = 'local')
dataset.preprocess(model_type="KmeansTM", min_word_length = 1)

2025-03-07 15:06:52.626 | INFO     | stream_topic.utils.dataset:fetch_dataset:121 - Fetching dataset: news_zh_hanlp
2025-03-07 15:06:52.628 | INFO     | stream_topic.utils.dataset:fetch_dataset:131 - Fetching dataset from local path
Preprocessing documents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7888/7888 [02:20<00:00, 56.05it/s]


In [24]:
metric = NPMI(dataset,language = "chinese", custom_stopwords=list(stopword)) #值越大越好   
scores = metric.score([["普京", "奥巴马"]])  #值越小越好
print("NPMI score:", scores)

471
{26, 4123, 6175, 4130, 4137, 4140, 45, 2095, 4145, 6194, 4148, 82, 2132, 2138, 4189, 104, 6248, 2155, 2156, 4208, 2161, 4209, 124, 2172, 145, 2193, 6292, 2200, 157, 167, 169, 2220, 6335, 194, 4291, 4292, 2252, 2253, 6350, 2259, 6360, 2267, 2268, 2270, 2281, 2284, 6391, 2302, 6410, 2321, 2330, 6426, 2333, 6435, 6439, 298, 4401, 6471, 2386, 2393, 4447, 4464, 6515, 4468, 6519, 4472, 4482, 406, 408, 412, 4511, 4512, 417, 421, 4529, 434, 6577, 6581, 438, 2486, 4554, 4555, 4564, 6612, 6617, 2530, 6629, 491, 492, 493, 496, 501, 4600, 4601, 505, 4602, 4603, 7430, 6666, 7431, 534, 4631, 4633, 551, 4648, 4662, 6710, 6712, 4665, 4667, 6715, 4675, 2632, 2636, 2646, 4695, 4703, 2656, 2659, 6760, 6762, 2682, 4739, 4740, 6799, 668, 669, 2725, 2726, 4775, 2730, 683, 2732, 4778, 697, 2749, 6853, 716, 2768, 2769, 727, 6871, 4828, 742, 6887, 746, 2795, 6891, 2798, 7476, 2803, 763, 2812, 766, 776, 4881, 2858, 4920, 4921, 4927, 835, 6993, 850, 4947, 4948, 6994, 874, 883, 886, 4983, 7030, 7031, 7035, 70

In [26]:
metric = PMI(dataset,language = "chinese", custom_stopwords=list(stopword)) #值越大越好   
scores = metric.score([["普京", "奥巴马"]])  #值越小越好
print("PMI score:", scores)

PMI score: 0.53458


In [7]:
def create_vocab_unsegmented(data, language,target_words=None, target_pairs=None):
        word_to_file = {}

        if language =="chinese":
            for file_num in range(0, len(data)):
                # 处理未分词数据，字符串匹配
                doc = data[file_num]
                doc = doc.strip()
                doc = re.sub(r"[^\u4e00-\u9fff\d]+", " ", doc)
                doc = re.sub(" +", " ", doc)
    
                if target_words:
                    # 遍历指定的目标词，检查每个目标词是否在文档中
                    for word in target_words:
                        # 使用字符串匹配检查目标词是否出现在文档中
                        if word in doc:
                            if word in word_to_file:
                                word_to_file[word].add(file_num)
                            else:
                                word_to_file[word] = {file_num}
                if target_pairs:
                    # 遍历指定的目标词对，检查每个目标词对是否在文档中
                    for pair in target_pairs:
                        word1, word2 = pair
                        if word1 in doc and word2 in doc:
                            if pair in word_to_file:
                                word_to_file[pair].add(file_num)
                            else:
                                word_to_file[pair] = {file_num}
        else:
            for file_num in range(0, len(data)):
                # 处理未分词数据，字符串匹配
                doc = data[file_num].lower()
                doc = doc.strip()
                doc = re.sub(r"[^a-zA-Z0-9]+\s*", " ", doc)
                doc = re.sub(" +", " ", doc)
    
                if target_words:
                    # 遍历指定的目标词，检查每个目标词是否在文档中
                    for word in target_words:
                        # 使用字符串匹配检查目标词是否出现在文档中
                        if word in doc:
                            if word in word_to_file:
                                word_to_file[word].add(file_num)
                            else:
                                word_to_file[word] = {file_num}
                if target_pairs:
                    # 遍历指定的目标词对，检查每个目标词对是否在文档中
                    for pair in target_pairs:
                        word1, word2 = pair
                        if word1 in doc and word2 in doc:
                            if pair in word_to_file:
                                word_to_file[pair].add(file_num)
                            else:
                                word_to_file[pair] = {file_num}

        return word_to_file

In [19]:
import re
def read_news_to_dataframe(file_path):
    """
    读取新闻文本文件并将每篇新闻报道分隔开来，返回一个 DataFrame。
    
    :param file_path: 新闻文本文件路径
    :return: 包含每篇新闻报道的 DataFrame
    """
    # 读取文件内容
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()

    # 使用正则表达式按照一个或多个空行分隔新闻报道
    news_list = re.split(r'\n\s*\n', content.strip())  # \n\s*\n 匹配一个或多个空行

    # 创建 DataFrame
    df = pd.DataFrame(news_list, columns=["text"])

    return df

# 假设文件路径是 'news.txt'
file_path = '/hongyi/stream/PMI/dataset/News-Commentary.zh'

# 调用函数
df = read_news_to_dataframe(file_path)
df['labels'] = range(len(df))
target_words = ["普京", "奥巴马"]
target_pairs = [("普京", "奥巴马")]
words_count = create_vocab_unsegmented(df['text'], "chinese", target_words=target_words)
pair_count = create_vocab_unsegmented(df['text'], "chinese", target_pairs=target_pairs)

In [15]:
import pandas as pd
import re
# 文件路径
file_path = "/hongyi/stream/dataset/cnews.val.txt"
# 创建一个空列表来存储处理后的数据
data = []
# 打开文件并读取每一行
with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        # 确保每一行至少有4个字符
        if len(line) > 3:
            # 提取前两个字符作为第一个元素
            first_two_chars = line[:2].strip()
            # 提取从第四个字开始的剩余部分作为第二个元素
            remaining_chars = line[3:].strip()
            # 将两个元素添加到列表中
            data.append([remaining_chars, first_two_chars])

# 创建 DataFrame
df = pd.DataFrame(data, columns=["text", "labels"])
target_words = ["普京", "奥巴马"]
target_pairs = [("普京", "奥巴马")]
words_count = create_vocab_unsegmented(df['text'], "chinese", target_words=target_words)
pair_count = create_vocab_unsegmented(df['text'], "chinese", target_pairs=target_pairs)

In [21]:
print(len(words_count["普京"]))
print(len(words_count["奥巴马"]))
print(len(pair_count[("普京", "奥巴马")]))

480
1110
114


In [22]:
import numpy as np
K = len(dataset.dataframe)
eps = 10 ** (-12)
pmi_w1w2 = np.log((len(pair_count[("普京", "奥巴马")]) * K) / ((len(words_count["普京"]) * len(words_count["奥巴马"])) + eps) + eps)
npmi_w1w2 = pmi_w1w2 / (-np.log((len(pair_count[("普京", "奥巴马")])) / K + eps))
print(pmi_w1w2)
print(npmi_w1w2)

0.523394946469243
0.12353253904543197


In [11]:
documents = list(dataset.dataframe.iloc[:,3:4]['text'].apply(lambda x: x.split()))
# documents = [item for sublist in documents for item in [sublist]*4]
K = len(documents)
l=[]
for doc in documents:
    l.append(len(doc))

In [25]:
# 构建词索引映射
index_mappings = build_index_mapping(documents)

# 查询的词对
word_pairs = [("普京", "奥巴马")]

# 计算非重叠出现次数
result_f = count_nonoverlapping_intervals(index_mappings, word_pairs)

# 设置单词间隔阈值
threshold = 50
# 计算非重叠出现次数
result_f_hat = count_nonoverlapping_intervals_with_threshold(index_mappings, word_pairs, threshold)

# 打印结果
for pair, counts in result_f.items():
    f = counts
    # print(f"词对 {pair}: {counts}")
# 打印结果
for pair, counts in result_f_hat.items():
    f_hat = counts
    # print(f"词对 {pair}: {counts}")


In [13]:
import numpy as np
def compute_hist_optimized(f, l, x, memo=None):
    """
    优化后的 ComputeHist 算法，利用动态规划加速
    :param f: 嵌入的非重叠出现次数
    :param l: 文档长度
    :param x: 跨度限制
    :param memo: 用于缓存中间结果的字典
    :return: 直方图分布 hist_f_l
    """
    if memo is None:
        memo = {}

    # 如果已经计算过，直接返回缓存结果
    if (f, l) in memo:
        return memo[(f, l)]

    # 初始化直方图 hist_f_l
    hist_f_l = np.zeros(f + 1, dtype=np.float64)

    # 边界条件
    if f > l:
        return hist_f_l  # 文档太短，无法嵌入
    if f == 0:
        hist_f_l[0] = 1  # 没有嵌入的可能
        memo[(f, l)] = hist_f_l
        return hist_f_l

    # 遍历文档中每个可能的 (i, j) 对
    for i in range(1,l):  # 从 1 到 l-1
        for j in range(i + 1, l+1):  # 从 i+1 到 l
            # 调用优化后的子问题计算
            hist_f_minus_1_l_minus_j = compute_hist_optimized(f - 1, l - j, x, memo)

            # 更新当前的直方图 hist_f_l
            for k in range(f):  # 遍历子问题结果
                if (j - i) < x:  # 跨度小于限制
                    hist_f_l[k + 1] += hist_f_minus_1_l_minus_j[k]
                else:  # 跨度大于等于限制
                    hist_f_l[k] += hist_f_minus_1_l_minus_j[k]

    # 缓存结果
    memo[(f, l)] = hist_f_l
    return hist_f_l


In [26]:
pi, pi2 = [], []
eplison = 0.7
for i in range(K):
    N_fl = compute_hist_optimized(f[i], l[i], threshold, memo=None)
    pi.append(sum(N_fl[f_hat[i]:])/sum(N_fl))
    for j in range(f_hat[i]+1):
        if sum(N_fl[j:])/sum(N_fl) < eplison:
            pi2.append(sum(N_fl[j:])/sum(N_fl))
            break

In [27]:
Z = sum(1 for prob in pi if prob < eplison)
print(Z)
EZ = sum(pi2)
print(EZ)

49
14.27843561620684


In [28]:
delta = 0.9
CSR = Z/(EZ+np.sqrt(-K*np.log(delta)/2))
print(CSR)

1.4135996430710243


In [29]:
CSA = sum(f_hat)/np.sqrt(K)
print(CSA)

0.8106792283998809


In [30]:
def count_word_occurrences(doc, word):
    return doc.count(word)
x_count, y_count = [], []
for doc in documents:
    x_count.append(count_word_occurrences(doc, word_pairs[0][0]))
    y_count.append(count_word_occurrences(doc, word_pairs[0][1]))
dx = sum(1 for num in x_count if num != 0)
dy = sum(1 for num in y_count if num != 0)
dxy = sum(1 for num in f_hat if num != 0)

In [31]:
cPMId=dxy/(dx*dy/K+np.sqrt(K)/(2*threshold)*np.sqrt(-np.log(delta)/2))
print(cPMId)

5.96340696975312


In [32]:
cPMIz = Z/(dx*dy/K+np.sqrt(K)/(2*threshold)*np.sqrt(-np.log(delta)/2))
print(cPMIz)

5.96340696975312
